# vulcan-retrieval quickstart

Two things this notebook demonstrates:

1. The differentiable forward model: `theta -> converged VULCAN-JAX chemistry -> ExoJax transit spectrum`, and pushing a forward-mode tangent (`jax.jvp`) through the whole chain.
2. Reading a finished SMC retrieval run from its `.npz` bundles.

**Prerequisites**: an editable install in a checkout of this repo (`pip install --no-deps -e .` from the repo root, with `vulcan-jax` installed) and the repo `data/` tree present. See the package README.

**Runtime**: the SMOKE profile below is the cheapest real configuration (CO-only, fully offline opacities, coarse 40-layer column), but it still runs the real chemistry solver to steady state. Expect a few minutes on a laptop CPU for the build cell, and roughly a minute per forward/jvp evaluation after that.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# import order is load-bearing: sensitivity imports forward.vulcan_chem (which
# fixes the VULCAN_JAX_* env vars and jax x64) BEFORE anything exojax. Importing
# exojax first raises a RuntimeError by design.
from retrieval_framework.forward import config
from retrieval_framework.forward.sensitivity import build_forward

import jax
import jax.numpy as jnp

## 1. Build the forward model (SMOKE profile)

`config.SMOKE` is the offline CO-band smoke configuration; `config.FULL` and `config.WIDE` are the science profiles (HITRAN downloads on first use, much heavier). The build converges the baseline chemistry once (the warm-up solve) and compiles the RT.

In [ ]:
m = build_forward(config.SMOKE)   # ~minutes: real chemistry warm-up + jit compile
wl_um = np.asarray(m.rt.wl_um)
print(f"native grid: {wl_um.size} points, {wl_um.min():.3f}-{wl_um.max():.3f} um")

## 2. Spectrum + end-to-end sensitivity via one jvp

`theta = [lnZ, C/O, lnKzz, dT]` relative to the baseline (`config.THETA0`). One `jax.jvp` call propagates a metallicity tangent through the converged chemistry AND the radiative transfer, so every wavelength gets `d(depth)/d(lnZ)`.

In [ ]:
theta0 = jnp.zeros(4)
e_lnZ = jnp.array([1.0, 0.0, 0.0, 0.0])
depth, ddepth_dlnZ = jax.jvp(m.forward, (theta0,), (e_lnZ,))
depth, ddepth_dlnZ = np.asarray(depth), np.asarray(ddepth_dlnZ)

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
order = np.argsort(wl_um)
ax0.plot(wl_um[order], depth[order] * 1e6, lw=0.7)
ax0.set_ylabel('transit depth (ppm)')
ax1.plot(wl_um[order], ddepth_dlnZ[order] * 1e6, lw=0.7, color='C3')
ax1.set_ylabel('d(depth)/d(lnZ) (ppm)')
ax1.set_xlabel('wavelength (um)')
fig.suptitle('WASP-39b SMOKE band: spectrum and metallicity sensitivity')
fig.tight_layout()

For the full 2x2 sensitivity figure over all four parameters, run `python examples/run_demo.py` from the repo root (writes `output/sensitivity.npz` and the manuscript figure). The wide-band (1-15 um) version is `run_figs.py`.

## 3. Reading a finished SMC retrieval run

A retrieval writes `.npz` bundles into `<run_dir>/data/<preset>/`. This cell reads whatever finished run it finds; if none exists yet, run the smoke retrieval first:

```bash
SMC_RETRIEVAL_PRESET=smoke python -m retrieval_framework.run_smc runs/w39b_smc_retrieval
```

(`python -m retrieval_framework.plot_smc <out_dir>` makes the full corner/spectrum/T-P figure set from the same files.)

In [ ]:
from pathlib import Path
import json

run_dir = Path.cwd().resolve()
if run_dir.name == 'examples':
    run_dir = run_dir.parent
candidates = sorted((run_dir / 'runs' / 'w39b_smc_retrieval' / 'data').glob('*/posterior_samples.npz'))
if not candidates:
    raise FileNotFoundError('no finished run found under runs/w39b_smc_retrieval/data/ '
                            '-- run the smoke retrieval first (see the cell above)')
samples_path = candidates[-1]
print('reading', samples_path)
cfg = json.loads((samples_path.parent / 'config.json').read_text())
names = cfg['inferred_param_names']
with np.load(samples_path) as z:
    print('arrays:', z.files)
    samples = z['samples'] if 'samples' in z.files else z[z.files[0]]
print('posterior cloud:', samples.shape)

fig, axes = plt.subplots(1, min(4, samples.shape[1]), figsize=(12, 2.5))
for i, ax in enumerate(np.atleast_1d(axes)):
    ax.hist(samples[:, i], bins=30, color='C0', alpha=0.8)
    ax.set_xlabel(names[i])
fig.tight_layout()